# Colab Setup
Run this notebook first in every session. It clones the repo, installs dependencies, checks GPU, and mounts Drive for persistent storage of checkpoints/outputs.

In [20]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
# 2. Clone the repo (replace with your actual GitHub URL once created)
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

Cloning into 'Review-Summarization-LLM'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 48 (delta 8), reused 32 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 29.30 KiB | 7.33 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/Review-Summarization-LLM/Review-Summarization-LLM


In [22]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [23]:
# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

CUDA available: True
Device: Tesla T4


In [24]:
import os
os.environ['KAGGLE_USERNAME'] = 'jsheffie'
os.environ['KAGGLE_KEY'] = 'KGAT_6307d48533c1425fe724f70bc0f3507d'

!python data/download_dataset.py

Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:00<00:00, 179MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [25]:
# 5. Set up Kaggle API for dataset download
# Upload your kaggle.json via the file browser on the left, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:00<00:00, 145MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [26]:
import pandas as pd
df = pd.read_csv('data/raw/Reviews.csv', nrows=5)
print(df.columns.tolist())
df.head()

['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [27]:
# 6. Run the preprocessing pipeline
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

2026-07-18 21:09:49,612 [INFO] Loaded 568454 rows from data/raw/Reviews.csv
2026-07-18 21:09:50,678 [INFO] drop_missing_and_nulls: 568454 -> 568454 rows
2026-07-18 21:09:52,040 [INFO] remove_duplicates: 568454 -> 567554 rows
2026-07-18 21:10:19,637 [INFO] filter_uninformative (min_words=5): 567554 -> 567534 rows
2026-07-18 21:34:28,813 [INFO] filter_non_english: 567534 -> 567254 rows
2026-07-18 21:34:29,045 [INFO] group_by_product (min=2): 74231 -> 43825 products, 536848 rows
2026-07-18 21:34:40,520 [INFO] Saved cleaned dataset to data/processed/clean_reviews.csv (536848 rows)
2026-07-18 21:34:40,687 [INFO] train_val_test_split: 35060/4382/4383 products -> 430166/54016/52666 rows
2026-07-18 21:35:08,916 [INFO] create_batches: 43825 product batches created
2026-07-18 21:35:11,045 [INFO] Saved 43825 product batches to data/processed/product_batches.json


In [34]:
!ls data/raw/

Reviews.csv  sample_reviews.csv


In [35]:
!ls -la data/processed/

total 674968
drwxr-xr-x 2 root root      4096 Jul 18 21:35 .
drwxr-xr-x 5 root root      4096 Jul 18 21:35 ..
-rw-r--r-- 1 root root 279134204 Jul 18 21:34 clean_reviews.csv
-rw-r--r-- 1 root root         0 Jul 18 21:09 .gitkeep
-rw-r--r-- 1 root root 132879897 Jul 18 21:35 product_batches.json
-rw-r--r-- 1 root root  27193208 Jul 18 21:34 test.csv
-rw-r--r-- 1 root root 223635839 Jul 18 21:34 train.csv
-rw-r--r-- 1 root root  28305357 Jul 18 21:34 val.csv


In [36]:
!pwd

/content/Review-Summarization-LLM/Review-Summarization-LLM


In [37]:
!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

Total reviews: 536848
Unique products: 43825

Missing values per column:
Id                         0
ProductId                  0
UserId                     0
ProfileName               25
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

Average review length: 79.9 words

Figures saved to outputs/figures


In [32]:
!mkdir -p /content/drive/MyDrive/Review-Summarization-LLM-backup
!cp -r data/processed /content/drive/MyDrive/Review-Summarization-LLM-backup/
!cp data/raw/Reviews.csv /content/drive/MyDrive/Review-Summarization-LLM-backup/

In [33]:
import pandas as pd
df = pd.read_csv('data/processed/clean_reviews.csv')
print(df[['ProductId', 'Summary', 'Text']].sample(5, random_state=1).to_string())

         ProductId                      Summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          Text
241326  B0029NVJX8         Excellent cat treats  I've been buying cat treats for many years. Of the brands and types that are still on the market, I find that Whiskas Temptations is the one treat that all cats love. They come in several flavors. My cats like the Hearty Beef and Turkey flavors best. Pricing can be an issue as I have noticed continual price increases in this product for the last few years. Shop around and ge